# Benchmark Qwen3 

| Model  | Model ID (Mặc định) |
|-------|---------------------|
| **Qwen3-4B** | `qwen3-4b` |
| **Qwen3-14B**  | `qwen/qwen-3-14b` |
| **Qwen3-32B**  | `qwen/qwen-3-32b` |
| **Qwen3-235B-A22B-Thinking**  | `qwen/qwen3-235b-a22b-thinking-2507` |

## 1. Cài đặt và Import

In [ ]:
import json
import os
import re
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.notebook import tqdm
import pandas as pd

## 2. Thiết lập Môi trường & Khởi tạo API Clients

In [2]:
load_dotenv()
DATA_FILE = Path("../data/bigdata_10_questions.json")
OUTPUT_DIR = Path("../outputs/qwen3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Khởi tạo OpenRouter Client
or_api_key = os.getenv("OPENROUTER_API_KEY")
or_base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

if not or_api_key:
    print("⚠️ Không tìm thấy OPENROUTER_API_KEY trong file .env!")
    or_client = None
else:
    or_client = OpenAI(base_url=or_base_url, api_key=or_api_key)

# Khởi tạo LM Studio Client (Local)
lm_base_url = os.getenv("LM_STUDIO_BASE_URL", "http://localhost:1234/v1")
lm_client = OpenAI(base_url=lm_base_url, api_key="lm-studio")

print("✅ Môi trường đã sẵn sàng.")

✅ Môi trường đã sẵn sàng.


## 3. Các hàm hỗ trợ sinh văn bản (Helper Functions)

Các hàm này sẽ gọi API, xử lý prompt, bóc tách suy luận (thinking) và tự động lưu kết quả ra file json.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior Big Data engineer and Spark expert. "
    "Answer accurately, practically, and follow the user's instructions. "
    "For code tasks, provide correct and production-aware code. "
    "Do not leave the answer unfinished."
)


def parse_generation_params(
    param_text: str | None,
    default_temperature: float = 0.6,
    default_max_completion_tokens: int = 3000,
    min_max_completion_tokens: int = 3000,
    max_completion_tokens_cap: int = 6000,
) -> tuple[float, int]:
    """
    Đọc recommended_generation_params nếu có.
    Quan trọng: ép max token tối thiểu 3000 để tránh output bị cắt quá sớm.
    """
    temp = default_temperature
    max_completion_tokens = default_max_completion_tokens

    if param_text:
        temp_match = re.search(r"temperature\s*=\s*([0-9.]+)", param_text)
        token_match = re.search(
            r"(?:max_tokens|max_completion_tokens)\s*=\s*(\d+)",
            param_text
        )

        if temp_match:
            temp = float(temp_match.group(1))

        if token_match:
            max_completion_tokens = int(token_match.group(1))

    max_completion_tokens = max(max_completion_tokens, min_max_completion_tokens)
    max_completion_tokens = min(max_completion_tokens, max_completion_tokens_cap)

    return temp, max_completion_tokens


def parse_thinking_content(response_text: str) -> tuple[str, str]:
    """
    Tách <think>...</think> nếu model trả thinking trong content.
    Với OpenRouter, reasoning có thể nằm ở message.reasoning,
    nên hàm này chỉ là fallback.
    """
    if not response_text:
        return "", ""

    response_text = str(response_text)

    think_match = re.search(
        r"<think>(.*?)</think>",
        response_text,
        flags=re.DOTALL | re.IGNORECASE
    )

    if think_match:
        thinking = think_match.group(1).strip()
        answer = re.sub(
            r"<think>.*?</think>",
            "",
            response_text,
            flags=re.DOTALL | re.IGNORECASE
        ).strip()
        return thinking, answer

    return "", response_text.strip()


def safe_model_dump(obj):
    """
    Chuyển response / usage object sang dict để lưu debug.
    Hỗ trợ Pydantic v2, Pydantic v1, dict, object thường.
    """
    if obj is None:
        return None

    if isinstance(obj, dict):
        return obj

    if hasattr(obj, "model_dump"):
        return obj.model_dump()

    if hasattr(obj, "dict"):
        return obj.dict()

    try:
        return dict(obj)
    except Exception:
        return str(obj)


def get_field(obj, field_name, default=None):
    """
    Lấy field an toàn từ object hoặc dict.
    """
    if obj is None:
        return default

    if isinstance(obj, dict):
        return obj.get(field_name, default)

    return getattr(obj, field_name, default)


def normalize_content(content) -> str:
    """
    Một số SDK/model có thể trả content dạng list.
    Hàm này ép về string để benchmark lưu ổn định.
    """
    if content is None:
        return ""

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(item))
        return "\n".join([p for p in parts if p])

    return str(content)


def looks_truncated(text: str) -> bool:
    """
    Heuristic phát hiện output bị cắt giữa chừng.
    Không hoàn hảo, nhưng hữu ích cho benchmark.
    """
    if not text:
        return False

    t = text.strip()

    # Markdown code fence bị mở nhưng chưa đóng
    if t.count("```") % 2 == 1:
        return True

    suspicious_endings = (
        "due to **",
        "because **",
        "using `",
        "using ``",
        "using `broadcast",
        "####",
        "###",
        "**",
        "-",
        ":",
        ",",
        "and",
        "or",
        "because",
        "due to",
        "such as",
        "for example",
    )

    lower_t = t.lower()
    return any(lower_t.endswith(x.lower()) for x in suspicious_endings)


def classify_response(
    model_output: str,
    full_response: str,
    thinking_content: str,
    finish_reason,
    usage=None,
    max_completion_tokens: int | None = None,
) -> list[str]:
    """
    Gắn warning để không còn trường hợp API 200 nhưng output rỗng vẫn bị coi là success.
    """
    warnings = []

    output = str(model_output or "").strip()
    full = str(full_response or "").strip()
    thinking = str(thinking_content or "").strip()

    if not output and not full:
        warnings.append("EMPTY_CONTENT_FROM_MODEL")

    if thinking and not output:
        warnings.append("REASONING_PRESENT_BUT_NO_FINAL_ANSWER")

    if finish_reason == "length":
        warnings.append("OUTPUT_TRUNCATED_BY_TOKEN_LIMIT")

    if finish_reason not in (None, "stop", "STOP"):
        warnings.append(f"NON_STOP_FINISH_REASON:{finish_reason}")

    if finish_reason is None:
        warnings.append("MISSING_FINISH_REASON")

    if output and len(output) < 30:
        warnings.append("VERY_SHORT_OUTPUT")

    if looks_truncated(output):
        warnings.append("TRUNCATED_OR_INCOMPLETE_OUTPUT")

    usage_dict = usage if isinstance(usage, dict) else {}
    completion_tokens = (
        usage_dict.get("completion_tokens")
        or usage_dict.get("completionTokens")
        or usage_dict.get("output_tokens")
    )

    if (
        completion_tokens is not None
        and max_completion_tokens is not None
        and completion_tokens >= int(max_completion_tokens * 0.95)
    ):
        warnings.append("COMPLETION_TOKENS_NEAR_LIMIT")

    return warnings


def call_completion_with_fallback(client: OpenAI, kwargs: dict):
    """
    Gọi API.
    Nếu SDK cũ không nhận max_completion_tokens thì fallback sang max_tokens.
    """
    try:
        return client.chat.completions.create(**kwargs)
    except TypeError as e:
        msg = str(e).lower()

        if "max_completion_tokens" in msg or "unexpected keyword" in msg:
            fallback_kwargs = dict(kwargs)
            fallback_kwargs["max_tokens"] = fallback_kwargs.pop("max_completion_tokens")
            return client.chat.completions.create(**fallback_kwargs)

        raise


def build_api_kwargs(
    model_id: str,
    prompt: str,
    provider: str,
    mode: str,
    temperature: float,
    max_completion_tokens: int,
    attempt: int = 1,
) -> dict:
    """
    Tạo kwargs gọi API cho từng provider/mode.
    """
    kwargs = {
        "model": model_id,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        "temperature": temperature,
        "max_completion_tokens": max_completion_tokens,
    }

    if provider == "openrouter":
        if mode == "thinking":
            # Attempt 1 cho reasoning 512 token.
            # Attempt 2 giảm reasoning xuống 256 token để chừa chỗ cho final answer.
            reasoning_budget = 512 if attempt == 1 else 256

            kwargs["extra_body"] = {
                "reasoning": {
                    "max_tokens": reasoning_budget,
                    "exclude": False,
                }
            }

        elif mode == "non-thinking":
            kwargs["extra_body"] = {
                "reasoning": {
                    "effort": "none",
                }
            }

    return kwargs


def extract_response_payload(response, max_completion_tokens: int):
    """
    Parse response thành các field benchmark cần lưu.
    """
    choices = get_field(response, "choices", []) or []

    if not choices:
        raise ValueError("API response has no choices")

    choice = choices[0]
    msg = get_field(choice, "message", None)

    if msg is None:
        raise ValueError("API response choice has no message")

    content = normalize_content(get_field(msg, "content", ""))

    # OpenRouter reasoning thường nằm ở message.reasoning.
    reasoning = (
        get_field(msg, "reasoning", None)
        or get_field(msg, "reasoning_content", None)
        or ""
    )

    reasoning = normalize_content(reasoning)

    reasoning_details = (
        get_field(msg, "reasoning_details", None)
        or get_field(msg, "reasoningDetails", None)
    )

    thinking_from_content, answer_from_content = parse_thinking_content(content)

    thinking_content = reasoning or thinking_from_content
    model_output = answer_from_content or content or ""
    full_response = content or ""

    finish_reason = get_field(choice, "finish_reason", None)
    native_finish_reason = get_field(choice, "native_finish_reason", None)

    usage = safe_model_dump(get_field(response, "usage", None))
    raw_response = safe_model_dump(response)

    warnings = classify_response(
        model_output=model_output,
        full_response=full_response,
        thinking_content=thinking_content,
        finish_reason=finish_reason,
        usage=usage,
        max_completion_tokens=max_completion_tokens,
    )

    return {
        "model_output": model_output,
        "thinking_content": thinking_content,
        "reasoning_details": reasoning_details,
        "full_response": full_response,
        "finish_reason": finish_reason,
        "native_finish_reason": native_finish_reason,
        "usage": usage,
        "warnings": warnings,
        "raw_response": raw_response,
    }


def should_retry(warnings: list[str]) -> bool:
    """
    Retry một lần nếu lỗi có khả năng do token/reasoning/truncation.
    """
    retryable = {
        "EMPTY_CONTENT_FROM_MODEL",
        "REASONING_PRESENT_BUT_NO_FINAL_ANSWER",
        "OUTPUT_TRUNCATED_BY_TOKEN_LIMIT",
        "TRUNCATED_OR_INCOMPLETE_OUTPUT",
        "COMPLETION_TOKENS_NEAR_LIMIT",
    }

    return any(w in retryable for w in warnings)


def run_and_save_benchmark(
    client: OpenAI,
    model_id: str,
    model_name: str,
    provider: str,
    mode: str = "thinking",
    limit: int = 0,
    sleep_sec: float = 2.0,
    retry_on_bad_output: bool = True,
):
    if not client:
        print(f"❌ Client cho {model_name} chưa được thiết lập. Bỏ qua.")
        return []

    with open(DATA_FILE, "r", encoding="utf-8") as f:
        samples = json.load(f)

    if limit > 0:
        samples = samples[:limit]

    results = []
    display_name = f"{model_name} ({mode})"

    print(f"\n🚀 Đang sinh dữ liệu cho: {display_name} [{model_id}] qua {provider}")
    print(f"📌 Số sample: {len(samples)}")

    for sample in tqdm(samples, desc=display_name):
        sample_id = sample.get("sample_id")
        prompt = sample.get("prompt", "")

        temp, max_completion_tokens = parse_generation_params(
            sample.get("recommended_generation_params")
        )

        started_at = datetime.now(timezone.utc).isoformat()
        start = time.perf_counter()

        final_payload = None
        final_error = None
        attempt_logs = []

        max_attempts = 2 if retry_on_bad_output else 1

        for attempt in range(1, max_attempts + 1):
            attempt_started_at = datetime.now(timezone.utc).isoformat()

            # Attempt 2 tăng token để giảm khả năng bị cắt output.
            attempt_max_completion_tokens = (
                max_completion_tokens if attempt == 1 else min(max_completion_tokens * 2, 8000)
            )

            try:
                kwargs = build_api_kwargs(
                    model_id=model_id,
                    prompt=prompt,
                    provider=provider,
                    mode=mode,
                    temperature=temp,
                    max_completion_tokens=attempt_max_completion_tokens,
                    attempt=attempt,
                )

                response = call_completion_with_fallback(client, kwargs)

                payload = extract_response_payload(
                    response=response,
                    max_completion_tokens=attempt_max_completion_tokens,
                )

                attempt_log = {
                    "attempt": attempt,
                    "started_at": attempt_started_at,
                    "ended_at": datetime.now(timezone.utc).isoformat(),
                    "max_completion_tokens": attempt_max_completion_tokens,
                    "finish_reason": payload.get("finish_reason"),
                    "warnings": payload.get("warnings", []),
                    "error": None,
                }

                attempt_logs.append(attempt_log)
                final_payload = payload
                final_error = None

                if retry_on_bad_output and attempt < max_attempts and should_retry(payload.get("warnings", [])):
                    print(
                        f"  🔁 Retry {sample_id}: "
                        f"{payload.get('warnings')} | "
                        f"finish_reason={payload.get('finish_reason')}"
                    )
                    time.sleep(1.0)
                    continue

                break

            except Exception as e:
                final_error = f"{type(e).__name__}: {str(e)}"

                attempt_logs.append({
                    "attempt": attempt,
                    "started_at": attempt_started_at,
                    "ended_at": datetime.now(timezone.utc).isoformat(),
                    "max_completion_tokens": attempt_max_completion_tokens,
                    "finish_reason": None,
                    "warnings": ["REQUEST_EXCEPTION"],
                    "error": final_error,
                    "traceback": traceback.format_exc(),
                })

                print(f"  ⚠️ Lỗi tại {sample_id}, attempt {attempt}: {final_error}")

                if attempt < max_attempts:
                    time.sleep(1.0)
                    continue

        latency_s = time.perf_counter() - start
        ended_at = datetime.now(timezone.utc).isoformat()

        if final_payload is None:
            final_payload = {
                "model_output": "",
                "thinking_content": "",
                "reasoning_details": None,
                "full_response": "",
                "finish_reason": None,
                "native_finish_reason": None,
                "usage": None,
                "warnings": ["REQUEST_EXCEPTION"],
                "raw_response": None,
            }

        warnings = final_payload.get("warnings", [])

        # Nếu có exception thật thì ưu tiên error exception.
        # Nếu API 200 nhưng output xấu thì error = warnings.
        if final_error:
            error = final_error
        elif warnings:
            error = ";".join(warnings)
        else:
            error = None

        results.append({
            "sample_id": sample_id,
            "model_name": model_name,
            "model_id": model_id,
            "provider": provider,
            "mode": mode,
            "prompt": prompt,

            "model_output": final_payload.get("model_output", ""),
            "thinking_content": final_payload.get("thinking_content", ""),
            "reasoning_details": final_payload.get("reasoning_details"),
            "full_response": final_payload.get("full_response", ""),

            "finish_reason": final_payload.get("finish_reason"),
            "native_finish_reason": final_payload.get("native_finish_reason"),
            "usage": final_payload.get("usage"),
            "warnings": warnings,

            "error": error,
            "latency_s": round(latency_s, 3),

            "metadata": {
                "started_at": started_at,
                "ended_at": ended_at,
                "attempts": attempt_logs,
            },

            # Chỉ lưu raw_response khi có warning/error để file không quá nặng.
            "raw_response": final_payload.get("raw_response") if error else None,
        })

        time.sleep(sleep_sec)

    fname = re.sub(r"[^A-Za-z0-9_.-]+", "_", display_name).strip("_") + "_outputs.json"
    output_path = OUTPUT_DIR / fname

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"✅ Đã lưu kết quả tại: {output_path}")

    df = pd.DataFrame(results)

    print("\n📊 Summary")
    print("Total:", len(df))
    print("Errors / warnings:", df["error"].notna().sum())
    print("Empty outputs:", (df["model_output"].fillna("").str.strip() == "").sum())

    if "finish_reason" in df.columns:
        display(
            df.groupby(["finish_reason", "error"], dropna=False)
              .size()
              .reset_index(name="count")
        )

    return results

## 4. Chạy Qwen3-4B 

In [ ]:
run_and_save_benchmark(
    client=lm_client,
    model_id="qwen3-4b", 
    model_name="Qwen3-4B",
    provider="lm_studio",
    mode="non-thinking",
    limit=0, # Thay bằng 1 hoặc 2 để test nhanh
)

## 5. Chạy Qwen3-14B 

In [ ]:
mode = "thinking"

results_qwen3_14b_test = run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen3-14b",
    model_name="Qwen3-14B",
    provider="openrouter",
    mode=mode,
    limit=1,
    sleep_sec=2.0,
    retry_on_bad_output=True,
)


🚀 Đang sinh dữ liệu cho: Qwen3-14B (thinking) [qwen/qwen3-14b] qua openrouter


Qwen3-14B (thinking):   0%|          | 0/10 [00:00<?, ?it/s]

✅ Đã lưu kết quả tại: ..\outputs\qwen3\Qwen3-14B_thinking_outputs.json


## 6. Chạy Qwen3-32B (OpenRouter)

In [ ]:
mode = "thinking"

results_qwen3_32b_test = run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen3-32b",
    model_name="Qwen3-32B",
    provider="openrouter",
    mode=mode,
    limit=1,
    sleep_sec=2.0,
    retry_on_bad_output=True,
)

## 7. Chạy Qwen3-235B-A22B-Thinking 

In [ ]:
results_qwen3_235b = run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen3-235b-a22b-thinking-2507",
    model_name="Qwen3-235B-A22B-Thinking",
    provider="openrouter",
    mode="thinking",
    limit=0,
    sleep_sec=2.0,
    retry_on_bad_output=True,
)

## 8. Xem thử dữ liệu đã sinh ra

In [ ]:
# Đọc kết quả từ các file output đã sinh ra
all_rows = []

for path in OUTPUT_DIR.glob("*_outputs.json"):
    try:
        rows = json.loads(path.read_text(encoding="utf-8"))
        for row in rows:
            row["_source_file"] = path.name
        all_rows.extend(rows)
    except Exception as e:
        print(f"Không đọc được {path}: {e}")

if not all_rows:
    print("Chưa có dữ liệu nào!")
else:
    df = pd.DataFrame(all_rows)

    # Thêm cột độ dài output để phát hiện output rỗng / quá ngắn
    df["output_len"] = df["model_output"].fillna("").astype(str).str.len()
    df["is_empty_output"] = df["model_output"].fillna("").astype(str).str.strip().eq("")

    show_cols = [
        "_source_file",
        "sample_id",
        "model_name",
        "mode",
        "provider",
        "finish_reason",
        "error",
        "warnings",
        "latency_s",
        "output_len",
        "is_empty_output",
    ]

    existing_cols = [c for c in show_cols if c in df.columns]

    display(df[existing_cols].head(50))

    print("Total rows:", len(df))
    print("Empty outputs:", df["is_empty_output"].sum())
    print("Rows with error/warning:", df["error"].notna().sum())

    print("\nFinish reason summary:")
    display(
        df.groupby(["finish_reason", "error"], dropna=False)
          .size()
          .reset_index(name="count")
          .sort_values("count", ascending=False)
    )

    print("\nRows cần kiểm tra:")
    problem_df = df[
        df["is_empty_output"]
        | df["error"].notna()
        | df["finish_reason"].fillna("").astype(str).ne("stop")
    ]

    display(problem_df[existing_cols].head(100))

### Checking one sample

In [ ]:
# Kiểm tra chi tiết sample test đầu tiên
if results_qwen3_14b_test:
    r = results_qwen3_14b_test[0]

    print("sample_id:", r.get("sample_id"))
    print("finish_reason:", r.get("finish_reason"))
    print("error:", r.get("error"))
    print("warnings:", r.get("warnings"))
    print("usage:", r.get("usage"))
    print("latency_s:", r.get("latency_s"))

    print("\n--- MODEL OUTPUT ---")
    print(r.get("model_output", ""))

    print("\n--- THINKING CONTENT LEN ---")
    print(len(r.get("thinking_content", "") or ""))

    print("\n--- ATTEMPTS ---")
    print(json.dumps(r.get("metadata", {}).get("attempts", []), ensure_ascii=False, indent=2))
else:
    print("Chưa có results_qwen3_14b_test")